# 语义内核

在这个代码示例中，您将使用 [语义内核](https://aka.ms/ai-agents-beginners/semantic-kernel) AI框架来创建一个基础代理。

本示例的目标是向您展示我们稍后在其他代码示例中实现不同代理模式时将使用的步骤。


## 导入所需的 Python 包


In [1]:
import json
import os

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## 创建客户端

在本示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 来访问 LLM。

`ai_model_id` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场中其他可用的模型，以查看不同的结果。

为了使用 `Azure Inference SDK`（用于 GitHub Models 的 `base_url`），我们将在 Semantic Kernel 中使用 `OpenAIChatCompletion` 连接器。此外，还有其他 [可用连接器](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion)，可以使用 Semantic Kernel 连接其他模型提供商。


In [2]:
import random   

# -----------------------------------------------------
# 1. 定义一个简单的代理工具 (Plugin)
# -----------------------------------------------------

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    # 使用 @kernel_function 装饰器，将这个方法暴露给 AI 模型作为工具
    # 使用 Annotated 为返回值添加详细的类型和描述，模型会利用这些信息来决定何时调用此函数
    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [3]:
# -----------------------------------------------------
# 2. 初始化配置和连接器
# -----------------------------------------------------
load_dotenv() # 从当前目录加载 .env 文件中的环境变量

# 使用通义大模型
model_name="qwen-max"
client = AsyncOpenAI(
    api_key=os.environ.get("DASHSCOPE_API_KEY"), 
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# 使用GPT大模型
# model_name = "gpt-4o-mini"
# # 创建 AsyncOpenAI 客户端实例
# client = AsyncOpenAI(
#     api_key=os.environ["GITHUB_TOKEN"],
#     base_url="https://models.inference.ai.azure.com/"
# )

# Create an AI Service that will be used by the `ChatCompletionAgent`
# 创建 AI 服务对象，作为 Semantic Kernel 与 Qwen-Max 模型通信的桥梁
chat_completion_service = OpenAIChatCompletion(
    ai_model_id=model_name,
    async_client=client,
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
# 创建 AI 服务对象，作为 Semantic Kernel 与 Qwen-Max 模型通信的桥梁
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="qwen-max",
    async_client=client,
)

## 创建代理

下面我们创建名为 `TravelAgent` 的代理。

在这个示例中，我们使用了非常简单的指令。您可以更改这些指令，观察代理如何做出不同的响应。

### 下方提示词翻译

您是一位乐于助人的 AI 智能体（Agent），能够帮助客户规划假期。

重要提示： 当用户指定目的地时，请始终针对该地点进行规划。只有在用户没有指定偏好时，才建议随机目的地。

对话开始时，请用以下信息介绍自己： "您好！我是您的 TravelAgent 助理。我可以帮您规划假期，并为您推荐有趣的旅行目的地。以下是您可以向我咨询的一些事项：
规划某一特定地点的一日游
建议一个随机的度假目的地
查找具有特定特色（海滩、山脉、历史遗迹等）的目的地
如果您不喜欢我的第一个建议，规划一次替代行程

今天您希望我帮您规划什么样的旅行呢？"

请始终优先考虑用户的偏好。如果他们提到了像“巴厘岛”或“巴黎”这样的具体目的地，请将您的规划重点放在该地点，而不是建议替代方案。


In [4]:
AGENT_INSTRUCTIONS = """You are a helpful AI Agent that can help plan vacations for customers.

Important: When users specify a destination, always plan for that location. Only suggest random destinations when the user hasn't specified a preference.

When the conversation begins, introduce yourself with this message:
"Hello! I'm your TravelAgent assistant. I can help plan vacations and suggest interesting destinations for you. Here are some things you can ask me:
1. Plan a day trip to a specific location
2. Suggest a random vacation destination
3. Find destinations with specific features (beaches, mountains, historical sites, etc.)
4. Plan an alternative trip if you don't like my first suggestion

What kind of trip would you like me to help you plan today?"

Always prioritize user preferences. If they mention a specific destination like "Bali" or "Paris," focus your planning on that location rather than suggesting alternatives.
"""

agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions=AGENT_INSTRUCTIONS,
)

## 运行代理

现在我们可以通过定义 `ChatHistory` 并将 `system_message` 添加到其中来运行代理。我们将使用之前定义的 `AGENT_INSTRUCTIONS`。

在这些定义完成后，我们创建一个 `user_inputs`，它代表用户发送给代理的内容。在这个例子中，我们将消息设置为 `Plan me a sunny vacation`。

你可以随意更改这条消息，看看代理会如何做出不同的回应。


In [5]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    # First, let the agent introduce itself
    agent_name = None
    full_response: list[str] = []
    function_calls: list[str] = []

    # Buffer to reconstruct streaming function call
    current_function_name = None
    argument_buffer = ""

    # Invoke agent with a simple greeting to trigger introduction
    async for response in agent.invoke_stream(
        messages="Hello",
        thread=thread,
    ):
        thread = response.thread
        agent_name = response.name
        content_items = list(response.items)

        for item in content_items:
            if isinstance(item, FunctionCallContent):
                if item.function_name:
                    current_function_name = item.function_name

                # Accumulate arguments (streamed in chunks)
                if isinstance(item.arguments, str):
                    argument_buffer += item.arguments
            elif isinstance(item, FunctionResultContent):
                # Finalize any pending function call before showing result
                if current_function_name:
                    formatted_args = argument_buffer.strip()
                    try:
                        parsed_args = json.loads(formatted_args)
                        formatted_args = json.dumps(parsed_args)
                    except Exception:
                        pass  # leave as raw string

                    function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                    current_function_name = None
                    argument_buffer = ""

                function_calls.append(f"\nFunction Result:\n\n{item.result}")
            elif isinstance(item, StreamingTextContent) and item.text:
                full_response.append(item.text)

    # Display agent introduction
    html_output = ""
    if function_calls:
        html_output += (
            "<div style='margin-bottom:10px'>"
            "<details>"
            "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
            "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
            "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
            f"{chr(10).join(function_calls)}"
            "</div></details></div>"
        )

    html_output += (
        "<div style='margin-bottom:20px'>"
        f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
        f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
    )

    display(HTML(html_output))

    # Now process user inputs
    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()


---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们对因使用此翻译而产生的任何误解或误读不承担责任。
